# Task 1 — EDA, Conversion Analysis & Trial Goal Definition

**Splendor Analytics Trial Activation Challenge**

This notebook explores the trial event data, identifies which activities are associated with conversion, and defines the Trial Goals that form the basis of the activation model.

## Section 1: Imports & Load Data

In [ ]:
# These are the libraries used in this analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
# Load the raw dataset
df = pd.read_csv(r"../data/DA_task.csv")
print(f"Raw shape: {df.shape}")

## Section 2: Data Cleaning

In [ ]:
# Parse all date columns from text to datetime
date_cols = ["TIMESTAMP", "CONVERTED_AT", "TRIAL_START", "TRIAL_END"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Lowercase column names for easier access
df.columns = df.columns.str.lower()

In [ ]:
# Remove duplicate rows
before = len(df)
df = df.drop_duplicates().reset_index(drop=True).copy()
after  = len(df)
print(f"Duplicates removed: {before - after:,}")
print(f"Clean shape:        {df.shape}")

In [ ]:
# Derive days_into_trial and within_trial flag
df["days_into_trial"] = (df["timestamp"] - df["trial_start"]).dt.days
df["within_trial"]    = (
    (df["timestamp"] >= df["trial_start"]) &
    (df["timestamp"] <= df["trial_end"])
)

# Filter to in-window events only
df_clean = df[df["within_trial"]].copy()
print(f"Final clean shape (in-window only): {df_clean.shape}")

In [ ]:
# Validation checks
print(f"Unique orgs:      {df_clean['organization_id'].nunique()}")
print(f"Converted orgs:   {df_clean[df_clean['converted']==True]['organization_id'].nunique()}")
print(f"Non-converted:    {df_clean[df_clean['converted']==False]['organization_id'].nunique()}")
print(f"Date range:       {df_clean['timestamp'].min()} -> {df_clean['timestamp'].max()}")

## Section 3: Build Org-Level Summary Table

In [ ]:
# Count how many times each org did each activity
activity_counts = (
    df_clean
    .groupby(["organization_id", "activity_name"])
    .size()
    .unstack(fill_value=0)
)

# Build core engagement metrics per org
org_stats = (
    df_clean
    .groupby("organization_id")
    .agg(
        converted         = ("converted",       "first"),
        converted_at      = ("converted_at",    "first"),
        trial_start       = ("trial_start",     "first"),
        trial_end         = ("trial_end",       "first"),
        total_events      = ("activity_name",   "count"),
        unique_activities = ("activity_name",   "nunique"),
        days_active       = ("days_into_trial", "nunique"),
        first_event_day   = ("days_into_trial", "min"),
        last_event_day    = ("days_into_trial", "max"),
    )
)

# Join activity counts to org stats
org_df = org_stats.join(activity_counts)

# Add modules_used score (0-5)
org_df["modules_used"] = (
    (org_df.get("Scheduling.Shift.Created",              0) > 0).astype(int) +
    (org_df.get("PunchClock.PunchedIn",                  0) > 0).astype(int) +
    (org_df.get("Absence.Request.Created",               0) > 0).astype(int) +
    (org_df.get("Timesheets.BulkApprove.Confirmed",      0) > 0).astype(int) +
    (org_df.get("Communication.Message.Created",         0) > 0).astype(int)
)

print(f"Org-level table shape: {org_df.shape}")
org_df.head(20)

## Section 4: Conversion Driver Analysis

In [ ]:
# Split into converted and non-converted groups
converted     = org_df[org_df["converted"] == True]
not_converted = org_df[org_df["converted"] == False]

print(f"Converted orgs:     {len(converted)}")
print(f"Non-converted orgs: {len(not_converted)}")
print(f"Conversion rate:    {len(converted)/len(org_df)*100:.1f}%")

In [ ]:
# Compare engagement metrics between groups
metrics = ["total_events", "unique_activities", "days_active",
           "first_event_day", "last_event_day", "modules_used"]

comparison = org_df.groupby("converted")[metrics].median()
print(comparison)

In [ ]:
# Boxplot comparison — converted (green) vs non-converted (red)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    sns.boxplot(
        data=org_df, x="converted", y=metric, ax=axes[i],
        hue="converted",
        palette={True: "#2ecc71", False: "#e74c3c"},
        legend=False
    )
    axes[i].set_title(metric.replace("_", " ").title())
    axes[i].set_xlabel("Converted")
    axes[i].set_ylabel("")

plt.suptitle("Converted vs Non-Converted: Engagement Metrics",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("engagement_comparison.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Mann-Whitney U Test — are the differences statistically significant?
print("\n--- Mann-Whitney U Test (p-value < 0.05 = real difference) ---")
for metric in metrics:
    group1 = converted[metric].dropna()
    group2 = not_converted[metric].dropna()
    stat, p = stats.mannwhitneyu(group1, group2, alternative="two-sided")
    significance = "SIGNIFICANT" if p < 0.05 else "not significant"
    print(f"{metric:<25} p={p:.4f}  {significance}")

In [ ]:
# Activity-level conversion rates
activity_cols = [col for col in org_df.columns
                 if any(col.startswith(x) for x in [
                     "Scheduling", "PunchClock", "Absence",
                     "Timesheets", "Communication", "Mobile",
                     "Shift", "Break", "Revenue", "Integration"])]

conversion_rates = []
for col in activity_cols:
    users = org_df[org_df[col] > 0]
    if len(users) >= 10:
        rate  = round(users["converted"].mean() * 100, 1)
        count = len(users)
        conversion_rates.append({"activity": col, "conversion_rate": rate, "orgs_used_it": count})

activity_conv_df = (pd.DataFrame(conversion_rates)
                      .sort_values("conversion_rate", ascending=False))

activity_conv_df["conversion_rate_num"] = activity_conv_df["conversion_rate"]
activity_conv_df["conversion_rate"] = activity_conv_df["conversion_rate"].apply(lambda x: f"{x}%")
print(activity_conv_df.to_string(index=False))

In [ ]:
# Bar chart of activity conversion rates
plt.figure(figsize=(12, 8))
sns.barplot(
    data=activity_conv_df,
    x="conversion_rate_num",
    y="activity",
    hue="activity",
    palette="RdYlGn_r",
    legend=False
)
plt.axvline(x=len(converted)/len(org_df)*100,
            color="black", linestyle="--", label="Overall conversion rate")
plt.title("Conversion Rate by Activity Usage", fontsize=13, fontweight="bold")
plt.xlabel("Conversion Rate (%)")
plt.ylabel("")
plt.legend()
plt.tight_layout()
plt.savefig("activity_conversion_rates.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Usage gap: converter % - non-converter % per activity
conv_usage     = (org_df[org_df["converted"]==True][activity_cols]  > 0).mean() * 100
non_conv_usage = (org_df[org_df["converted"]==False][activity_cols] > 0).mean() * 100

usage_gap = pd.DataFrame({
    "converters_%":     conv_usage,
    "non_converters_%": non_conv_usage,
    "gap":              conv_usage - non_conv_usage
}).sort_values("gap", ascending=False).round(1)

print(usage_gap.to_string())

## Section 5: Trial Goal Definition

### Approach
To discover which activities are indicative of conversion, two complementary analytical methods were used:

1. **Engagement metric comparison** — compared converted vs non-converted organisations across key behavioural metrics using Mann-Whitney U statistical tests.
2. **Activity-level conversion rate analysis** — for each of the 28 activities, calculated the conversion rate among organisations that used that activity vs the 21.3% baseline.

The statistical tests showed no significant difference in general engagement metrics (all p > 0.05). The activity-level analysis identified three activities consistently associated with above-baseline conversion rates.

### Defined Trial Goals

| Goal | Condition | Evidence |
|---|---|---|
| **Goal 1 — Core Scheduling Activated** | `Scheduling.Shift.Created >= 3` | 21.8% conv rate, +2.6pp usage gap |
| **Goal 2 — Scheduling Depth Reached** | `Scheduling.Template.ApplyModal.Applied >= 1` | 25.0% conv rate, +2.4pp usage gap |
| **Goal 3 — Time & Attendance Activated** | `PunchClock.PunchedIn >= 1` | 22.7% conv rate, +1.9pp usage gap |

**Trial Activation** = all 3 goals completed within the 30-day trial window.

In [ ]:
# Apply Trial Goals
goal1 = org_df["Scheduling.Shift.Created"]              >= 3
goal2 = org_df.get("Scheduling.Template.ApplyModal.Applied", 0) >= 1
goal3 = org_df.get("PunchClock.PunchedIn",               0) >= 1

org_df["goal1_met"]       = goal1
org_df["goal2_met"]       = goal2
org_df["goal3_met"]       = goal3
org_df["trial_activated"] = goal1 & goal2 & goal3

In [ ]:
# Validation
print("--- Final Goal Completion Rates ---")
for i, goal in enumerate(["goal1_met","goal2_met","goal3_met"], 1):
    total = org_df[goal].sum()
    pct   = org_df[goal].mean() * 100
    print(f"Goal {i}: {total} orgs ({pct:.1f}%)")

activated = org_df["trial_activated"].sum()
total     = len(org_df)
print(f"\nTrial Activated: {activated} orgs ({activated/total*100:.1f}%)")

print("\n--- Activation Rate by Conversion Status ---")
result = org_df.groupby("converted")["trial_activated"].mean() * 100
for status, rate in result.items():
    label = "Converters" if status == True else "Non-converters"
    print(f"{label}: {rate:.1f}%")

# Export org summary
org_df.to_csv("org_summary.csv", index=True)
print("\nExported successfully.")